In [13]:
# Sampling the data for training and out-of-time testing

import pandas as pd
import numpy as np

df_raw = pd.read_csv("data/abt_churn.csv")

#defining the df we are using for the model
filtro = df_raw["dtRef"] != '2025-04-01'
filtro_1 = df_raw["dtRef"] == '2025-04-01'

df_train = df_raw[filtro].copy().reset_index(drop=True)
oot = df_raw[filtro_1].copy().reset_index(drop=True) # out of time

X = df_train.drop(columns=["flagChurn", "dtRef", "idUsuario"]).copy()
y = df_train["flagChurn"].copy()


In [14]:
# Train-test split

from sklearn import model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.mean(), y_test.mean(), oot["flagChurn"].mean())

0.46894559460760715 0.4687199230028874 0.504950495049505


In [15]:
# Model pipeline

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, roc_auc_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

processor = StandardScaler()

pipelines = {
    "Logistic Regression": Pipeline([
        ("Process", processor),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Naive Bayes": Pipeline([
        ("Process", processor),
        ("model", GaussianNB())
    ]),
    "Decision Tree": Pipeline([
        ("Process", processor),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
    ])
}

In [16]:
# Train and evaluate models

for name, pipe in pipelines.items():
    print(f"\n===== {name} =====")
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:,1] if hasattr(pipe, "predict_proba") else None

    print(classification_report(y_test, y_pred, digits=3))
    if y_prob is not None:
        auc = roc_auc_score(y_test, y_prob)
        print(f"ROC-AUC: {auc:.3f}")


===== Logistic Regression =====
              precision    recall  f1-score   support

           0      0.811     0.692     0.747       552
           1      0.701     0.817     0.755       487

    accuracy                          0.751      1039
   macro avg      0.756     0.755     0.751      1039
weighted avg      0.759     0.751     0.750      1039

ROC-AUC: 0.830

===== Naive Bayes =====
              precision    recall  f1-score   support

           0      0.881     0.361     0.512       552
           1      0.566     0.945     0.708       487

    accuracy                          0.634      1039
   macro avg      0.723     0.653     0.610      1039
weighted avg      0.733     0.634     0.603      1039

ROC-AUC: 0.713

===== Decision Tree =====
              precision    recall  f1-score   support

           0      0.768     0.754     0.761       552
           1      0.726     0.741     0.734       487

    accuracy                          0.748      1039
   macro avg 